# HStream Extractor (Colab)

Requires **hanime-plugin**. **4K preferred** with fallback.

Subtitles: **scrape live .ass URL from the episode page** (CDN hosts rotate), then known hosts.

Credits: [hanime-plugin](https://github.com/cynthia2006/hanime-plugin)


In [ ]:
import os, subprocess, requests, glob, re
from tqdm.notebook import tqdm
print('Installing...')
subprocess.run(['pip','install','-q','--upgrade','yt-dlp','requests','tqdm','hanime-plugin'], check=False)
subprocess.run(['apt-get','update','-qq'], check=False)
subprocess.run(['apt-get','install','-y','-qq','aria2','ffmpeg'], check=False)
if subprocess.run(['which','deno'], capture_output=True).returncode != 0:
    subprocess.run('curl -fsSL https://deno.land/install.sh | sh', shell=True, check=False)
    os.environ['PATH'] = os.path.expanduser('~/.deno/bin') + os.pathsep + os.environ.get('PATH','')
print('OK')


In [ ]:
# @title Settings
URL_LIST = "https://hstream.moe/hentai/hamehara-sore-sekuhara-desu-1"  #@param {type:"string"}
DESTINATION_FOLDER = "/content/downloads"  #@param {type:"string"}
XSRF_TOKEN = ""  #@param {type:"string"}
HSTREAM_SESSION = ""  #@param {type:"string"}
SERIES_SLUG = ""  #@param {type:"string"}
YEAR = "2026"  #@param {type:"string"}
MAKE_SAMPLE = True  #@param {type:"boolean"}
SAMPLE_START = "00:12:01"  #@param {type:"string"}
SAMPLE_DURATION_SEC = 60  #@param {type:"integer"}
print('Settings loaded')


In [ ]:
os.makedirs(DESTINATION_FOLDER, exist_ok=True)
deno_bin = os.path.expanduser("~/.deno/bin")
if os.path.isdir(deno_bin):
    os.environ["PATH"] = deno_bin + os.pathsep + os.environ.get("PATH", "")

urls = [u.strip() for u in URL_LIST.replace("\n", " ").split() if u.strip()]
print(f"Found {len(urls)} links\n")

cookie_parts = []
if XSRF_TOKEN.strip():
    cookie_parts.append(f"XSRF-TOKEN={XSRF_TOKEN.strip()}")
if HSTREAM_SESSION.strip():
    cookie_parts.append(f"hstream_session={HSTREAM_SESSION.strip()}")
COOKIE_HEADER = "; ".join(cookie_parts)

SUB_HOSTS = [
    "https://oppai-str.shoujo-h.org",
    "https://imoto-str.ane-h.xyz",
    "https://shinobu-str.rorikon-h.xyz",
]
yp = YEAR.strip() or "2024"
YEARS = []
for y in (yp, "2026", "2025", "2024", "2023", "2022", "2021"):
    if y not in YEARS:
        YEARS.append(y)
FORMAT_TRIES = ["best", "bestvideo*+bestaudio/best", "best[height<=2160]", "best[height<=1080]", "best[height<=720]"]

def resolve_subtitle_from_page(page_url):
    headers = {"User-Agent": "Mozilla/5.0", "Referer": "https://hstream.moe/"}
    if COOKIE_HEADER:
        headers["Cookie"] = COOKIE_HEADER
    try:
        r = requests.get(page_url, headers=headers, timeout=30)
        if r.status_code != 200:
            tqdm.write(f"  page HTTP {r.status_code}")
            return None
        found = []
        for pat in [r'href=["\'](https?://[^"\']+?/eng\.ass)["\']', r'href=["\'](https?://[^"\']+?\.ass)["\']']:
            for m in re.finditer(pat, r.text, re.I):
                if m.group(1) not in found:
                    found.append(m.group(1))
        if not found:
            tqdm.write("  no .ass on page")
            return None
        for u in found:
            if "eng.ass" in u.lower():
                tqdm.write(f"  page subtitle: {u}")
                return u
        tqdm.write(f"  page subtitle: {found[0]}")
        return found[0]
    except Exception as e:
        tqdm.write(f"  scrape failed: {e}")
        return None

def try_download_sub(sub_url, sub_path):
    try:
        r = requests.get(sub_url, stream=True, timeout=30)
        if r.status_code != 200:
            return False
        total = int(r.headers.get("content-length", 0))
        with open(sub_path, "wb") as f, tqdm(desc="Subtitle", total=total, unit="B", unit_scale=True, unit_divisor=1024, leave=False) as bar:
            for chunk in r.iter_content(1024):
                bar.update(len(chunk)); f.write(chunk)
        return True
    except Exception:
        return False

def run_ytdlp(url, out, fmt, dl):
    cmd = ["yt-dlp", "-f", fmt, "--downloader", dl, "--concurrent-fragments", "8", "-o", out, "--no-mtime", "--retries", "5", "--fragment-retries", "5"]
    if dl == "aria2c":
        cmd += ["--downloader-args", "aria2c:-x 16 -s 16 -k 1M"]
    if COOKIE_HEADER:
        cmd += ["--add-header", f"Cookie: {COOKIE_HEADER}"]
    cmd.append(url)
    return subprocess.run(cmd, check=True, capture_output=True, text=True)

for i, url in enumerate(tqdm(urls, desc="Overall", unit="video"), 1):
    tqdm.write(f"\n[{i}/{len(urls)}] {url}")
    out = os.path.join(DESTINATION_FOLDER, "%(title)s.%(ext)s")
    ok = False
    for fmt in FORMAT_TRIES:
        for dl in ("aria2c", "ffmpeg"):
            try:
                tqdm.write(f"  try {fmt} / {dl}")
                run_ytdlp(url, out, fmt, dl)
                ok = True
                break
            except subprocess.CalledProcessError as e:
                err = e.stderr or e.stdout or ""
                for line in err.strip().splitlines()[-6:]:
                    if "ERROR" in line or "404" in line:
                        tqdm.write(line)
        if ok:
            break
    if not ok:
        tqdm.write("download failed"); continue

    files = [f for f in glob.glob(os.path.join(DESTINATION_FOLDER, "*")) if not f.endswith(".ass") and "-sample" not in f.lower()]
    if not files:
        continue
    latest = max(files, key=os.path.getctime)
    base = os.path.splitext(os.path.basename(latest))[0]
    final_mkv = os.path.join(DESTINATION_FOLDER, f"{base}.mkv")
    if latest.endswith(".mkv"):
        tqdm.write(f"Already MKV: {latest}"); continue

    ep = url.rstrip("/").split("/")[-1].split("-")[-1]
    slug = "-".join(url.rstrip("/").split("/")[-1].split("-")[:-1])
    sub_path = os.path.join(DESTINATION_FOLDER, f"{base}.ass")
    sub_ok = False

    live = resolve_subtitle_from_page(url)
    if live and try_download_sub(live, sub_path):
        sub_ok = True

    if not sub_ok:
        cands = []
        if SERIES_SLUG.strip():
            cands.append(SERIES_SLUG.strip())
        cands += [slug.replace("-", "."), slug, ".".join(w.capitalize() for w in slug.split("-"))]
        cands = list(dict.fromkeys(cands))
        for host in SUB_HOSTS:
            for y in YEARS:
                for s in cands:
                    su = f"{host}/{y}/{s}/E{int(ep):02d}/eng.ass"
                    tqdm.write(f"  try {su}")
                    if try_download_sub(su, sub_path):
                        sub_ok = True; break
                if sub_ok: break
            if sub_ok: break

    if sub_ok:
        try:
            subprocess.run(["ffmpeg","-y","-i",latest,"-i",sub_path,"-map","0","-map","1","-c","copy","-metadata:s:s:0","language=eng",final_mkv], check=True)
            if os.path.exists(sub_path): os.remove(sub_path)
            if latest != final_mkv and os.path.exists(latest): os.remove(latest)
            tqdm.write(f"Saved: {final_mkv}")
        except Exception as ex:
            tqdm.write(f"remux error: {ex}")
    else:
        tqdm.write("No subtitle; kept original")

print("\nDONE")


In [ ]:
# @title Samples
from pathlib import Path
def parse_ts(ts):
    p=[int(x) for x in ts.strip().split(':')]
    return p[0]*3600+p[1]*60+p[2] if len(p)==3 else (p[0]*60+p[1] if len(p)==2 else p[0])
def fmt_ts(t):
    h,r=divmod(max(0,t),3600); m,s=divmod(r,60)
    return f'{h:02d}:{m:02d}:{s:02d}' if h else f'{m:02d}:{s:02d}'
if not MAKE_SAMPLE:
    print('skip samples')
else:
    dest=Path(DESTINATION_FOLDER); st=parse_ts(SAMPLE_START); dur=int(SAMPLE_DURATION_SEC)
    sl,el=fmt_ts(st),fmt_ts(st+dur); mins=max(1,round(dur/60))
    vids=sorted([p for p in dest.iterdir() if p.suffix.lower() in {'.mkv','.mp4','.webm'} and '-sample' not in p.stem.lower()])
    for v in vids:
        out=dest/f"{v.stem}-sample [{sl} - {el}] {mins} Minute{v.suffix}"
        print(v.name,'->',out.name)
        try:
            subprocess.run(['ffmpeg','-y','-ss',str(st),'-i',str(v),'-t',str(dur),'-c','copy',str(out)],check=True,capture_output=True)
            print(' OK')
        except Exception as e:
            print(' fail',e)


In [ ]:
!zip -r /content/hstream_downloads.zip {DESTINATION_FOLDER}
print('zip ready')
